Notebook: Descarga de genomas SARS-CoV-2 secuenciados desde México entre 2020-2024

# Imports y configuración

In [1]:
from Bio import Entrez, SeqIO
import pandas as pd
import os
import random
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:

def solicitar_correo():
    """
    Pide el correo electrónico que NCBI Entrez demanda para identificar al usuario
    y lo guarda en Entrez.email. Repite la pregunta hasta recibir un texto con forma de correo
    (con "@" y un punto después de la arroba; por ejemplo, @gmail.com sería una terminación válida).

    Parámetros:
        Ninguno.

    Regresa:
        El correo introducido (str).

    """
    while True:
        try:
            correo = input("Por favor, ingresa tu correo electrónico para usar NCBI Entrez: ").strip()
        # Contempla que la consola no está funcionando.    
        except EOFError as e:
            raise ValueError("No se pudo leer el correo porque no hay entrada disponible.") from e

        if "@" in correo and "." in correo.split("@")[-1]:
            break
        print("El correo no parece válido (ejemplo: nombre@dominio.com). Intenta de nuevo.")

    Entrez.email = correo
    print(f"Correo electrónico introducido: {Entrez.email}")
    return correo

# Búsqueda de genomas de México por año (Entrez esearch)

In [3]:
def buscar_genomas_mexico(year):

    """
    Busca en la base de datos Nucleotide de NCBI los genomas completos de SARS-CoV-2
    de México recolectados en un año.

    Parámetros:
        year: Año de recolección (int o str), por ejemplo, 2021.

    Regresa:
        Lista (list de str) con los IDs de NCBI de los genomas encontrados (máximo 10,000).
        Puede estar vacía si la búsqueda no encontró nada.

    """
    consulta = f'SARS-CoV-2[Organism] AND Mexico[Location] AND {year}[Collection Date] AND complete genome[Title]'

    try:
        handle = Entrez.esearch(db="nucleotide", term=consulta, retmax=10000)
        resultado = Entrez.read(handle)
        handle.close()
    except RuntimeError as e:
        raise ConnectionError(f"No se pudo consultar Entrez para {year}: {e}") from e
    
    return resultado["IdList"]



# Descargar los 80 genomas por año con Entrez efetch

In [4]:
def descargar_genomas(year, n=80):
    """
        Descarga n genomas aleatorios de México para un año y los guarda en un archivo FASTA
        (almacenado en ../data/raw/ncbi_<año>.fasta). 
    
        Parámetros:
            year: Año de recolección (int).
            n: Número de genomas a descargar (int). Si hay menos disponibles, se descargan todos.
    
        Regresa:
            La ruta (str) del archivo FASTA creado.
    
    """

    if not isinstance(n, int) or n <= 0:
        raise ValueError(f"n debe ser un entero positivo, pero se recibió {n}.")

    ids = buscar_genomas_mexico(year)
    if not ids:
        raise ValueError(f"Advetencia: la búsqueda para el año {year}: no regresó genomas.")

    if len(ids) < n:
        print(f"Adverencia: para el año {year} solo hubo {len(ids)} genomas, así que se descargarán todos.")
        seleccionados = ids
    else:
        random.seed(42)  # Esta semilla se ocupa para garantizar la reproducibilidad de la selección aleatoria.
        seleccionados = random.sample(ids, n)

    try:
        handle = Entrez.efetch(db="nucleotide", id=",".join(seleccionados), rettype="fasta", retmode="text")
        fasta = handle.read()
        handle.close()
        
        os.makedirs("../data/raw/ncbi", exist_ok=True)
        archivo = f"../data/raw/ncbi_{year}.fasta"
        with open(archivo, "w") as f:
            f.write(fasta)
        
        print(f"{year}: {len(seleccionados)} genomas guardados en {archivo}")
        return archivo

    except ConnectionError as e:
        raise ConnectionError(f"Hubo un error de conexión y se pudieron descargar los genomas de {year}: {e}") from e


# Descargar la referencia de Wuhan (NC_045512.2)

In [19]:

def descargar_referencia():
    """
    Descarga el genoma de referencia del SARS-COV-2, secuenciado en Wuhan (su identificador es NC_045512.2).

    Parámetros:
        Ninguno.

    Regresa:
        El registro de la referencia (Bio.SeqRecord.SeqRecord).

    """
    
    try:
            
        handle = Entrez.efetch(db="nucleotide", id="NC_045512.2", rettype="fasta", retmode="text")
        referencia = handle.read()
        handle.close()
               
    except ConnectionError as e:
        raise ConnectionError(f"No se pudo descargar la referencia NC_045512.2: {e}") from e

    os.makedirs("../data/raw/reference", exist_ok=True)
    with open("../data/raw/reference/wuhan_ref.fasta", "w") as f:
        f.write(referencia)

    return SeqIO.read("../data/raw/reference/wuhan_ref.fasta", "fasta")

# Análisis preliminar de todos los genomas descargados

In [14]:
def generar_datos_por_genoma(anios=[2020, 2021, 2022, 2023, 2024]):
    """
    Lee los FASTA descargados y calcula, para cada genoma, su longitud, el porcentaje de bases
    ambiguas "N" y el contenido de GC.

    Parámetros:
        anios: Lista de años que se tomarán en cuenta (list of int).

    Regresa:
        DataFrame con una fila por genoma y las columnas anio, id, longitud, pct_N y pct_GC.

    """
    filas = []
    for year in anios:
        archivo = f"../data/raw/ncbi_{year}.fasta"
        try:
            seqs = list(SeqIO.parse(archivo, "fasta"))
        except FileNotFoundError:
            print(f"Advertencia: no hay ningún archivo fasta para {year}; ese año se excluirá del análisis.")
            continue
        if not seqs:
            print(f"Advertencia: el archivo de {year} no contiene secuencias; no se tomará en cuenta.")
            continue

        for s in seqs:
            seq = str(s.seq).upper()
            if not seq:
                continue
            filas.append({
                "anio": year,
                "id": s.id,
                "longitud": len(seq),
                "pct_N": seq.count("N") / len(seq) * 100,
                "pct_GC": (seq.count("G") + seq.count("C")) / len(seq) * 100 ,
            })

    if not filas:
        raise ValueError("No hay genomas para analizar.")
    return pd.DataFrame(filas)

# Descripción estadística de los datos

In [39]:
def resumen_estadistico_anual(df_genomas):
    """
    Obtiene los datos estadísticos de los genomas descargados, agrupándolos por año.

    Parámetros:
        df_genomas: DataFrame producido por generar_datos_por_genoma.

    Regresa:
        resumen_anual: DataFrame que tiene columnas adicionales, por año, sobre el número de genomas
        descargados; su longitud promedio, mínima y máxima; y los porcentajes promedio del número de
        bases ambiguas (N) y el contenido de GC.
    
    """

    try:
        resumen_anual = df_genomas.groupby("anio").agg(n_genomas=("id", "count"),
        longitud_promedio=("longitud", "mean"), longitud_min=("longitud", "min"),
        longitud_max=("longitud", "max"), pct_GC_promedio=("pct_GC", "mean"),
        pct_N_promedio=("pct_N", "mean"),).round(3).reset_index()


    except KeyError as e:
        raise ValueError(f"Falta la columna {e} en el DataFrame original de todos los genomas.") from e

    print("\nResumen estadístico anual (con redondeo a tres decimales):")
    print(resumen_anual.to_string(index=False))
    return resumen_anual


# Visualización de los datos 

In [47]:
def graficar_boxplots_scatterplot(df_genomas):
    """
    Dibuja tres gráficas: boxplot del contenido GC por año, boxplot del porcentaje de N por año
    y diagrama de dispersión de GC frente a N (un punto por genoma, coloreado por año).

    Parámetros:
        df_genomas: DataFrame producido por generar_datos_por_genoma.

    Regresa:
        None. La figura se muestra en pantalla.

    
    """
    try: 
        if df_genomas.empty or not {"anio", "pct_GC", "pct_N", "longitud"}.issubset(df_genomas.columns):
            raise ValueError("No hay datos suficientes para graficar.")
    
        orden = sorted(df_genomas["anio"].unique())
        fig, ejes = plt.subplots(1, 3, figsize=(15,4.5))
        
        for eje, col, etq, color in [(ejes[0], "pct_GC", "Porcentaje del contenido GC", "#2a78d6"), (ejes[1], "pct_N", "Porcentaje de bases indefinidas (N)", "#7570b3")]:
            sns.boxplot(data=df_genomas, x="anio", y=col, order=orden, color=color, ax=eje)
            eje.set(xlabel="Año", ylabel=etq, title=f"{etq} por año")

        scatter_hues = ["#adad1a", "#8bc4Bb", "#334e9e", "#4bab35", "#0d1306"]
        sns.scatterplot(data=df_genomas, x="pct_GC", y="pct_N", hue=df_genomas["anio"].rename("Año"), palette=scatter_hues, ax=ejes[2])
        ejes[2].set(xlabel="Porcentaje de GC", ylabel="Porcentaje de bases indefinidas (N)", title="Dispersión entre los porcentajes de GC y las bases indefinidas")
        fig.tight_layout()
        
        
        os.makedirs("../data/processed", exist_ok=True)
        plt.savefig("../data/processed/boxplots_scatterplots_generales.png", dpi=300, bbox_inches="tight")
    except OSError as e:
        print(f"No se pudo crear o guardar la figura, o se presentaron ambos problemas. Ocurrió este error: {e}")
    plt.show()

In [41]:
def  generar_grafica_barras(df_estadistico):
    """
    Dibuja una gráfica de barras que compara la longitud
    promedio documentada en los 5 años que comprende el análisis.

    Parámetros:
        df: DataFrame producido por resumen_estadistico_anual.

    Regresa:
        None. La figura se muestra y se resguarda (si no se puede almacenar, el usuario es notificado).

    """
    try: 
        columnas = {"longitud_promedio", "anio"}
        if df_estadistico.empty or not columnas.issubset(df_estadistico.columns):
            raise ValueError("No existe la columna que se requiere para hacer esta gráfica.")
    
        fig, axes = plt.subplots(figsize=(8, 4))
        axes.bar(df_estadistico["anio"].astype(str), df_estadistico["longitud_promedio"], color="#55A868")
        axes.set_title("Longitud promedio de los genomas descargados por año")
        axes.set_xlabel("Año")
        axes.set_ylabel("Longitud (en bp)")
    
        plt.tight_layout()
        os.makedirs("../data/processed", exist_ok=True)
        plt.savefig("../data/processed/tendencias_genomas_longitud.png", dpi=300, bbox_inches="tight")
    except OSError as e:
        print(f"No se pudo guardar la figura: {e}")
    plt.show()

# Bloque para ejecutar el código

In [ ]:

if __name__ == "__main__":
    solicitar_correo()
    ref= descargar_referencia()
    print(f"La referencia con ID {ref.id} tiene {len(ref.seq)} nucleótidos, proveniente de Wuhan, se descargó con éxito.")
    for year in [2020, 2021, 2022, 2023, 2024]:
        ids = buscar_genomas_mexico(year)
        print(f"Para el año {year}, se encontraron {len(ids)} genomas.")
        descargar_genomas(year, n=80) 
        
    df_genomas = generar_datos_por_genoma(anios=[2020, 2021, 2022, 2023, 2024])
    df_estadistico = resumen_estadistico_anual(df_genomas)
    graficar_boxplots_scatterplot(df_genomas)
    generar_grafica_barras(df_estadistico)
    